# CP101 Project - Group 18


## SECTION 1 - LODES data cleaning

### Step 0 - import libraries & set up directory paths

In [1]:
#import libraries
import gc # garbage collector for RAM conservation
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
# folder paths
lodes_path = "../data/raw/lodes"
boundaries_path = "../data/raw/boundaries"
processed_path = "../data/processed"

# 9-County Bay Area 5-digit FIPS codes (State '06' + 3-digit County)
bay_area_county_FIPS = [
    "06001",  # Alameda
    "06013",  # Contra Costa
    "06041",  # Marin
    "06055",  # Napa
    "06075",  # San Francisco
    "06081",  # San Mateo
    "06085",  # Santa Clara
    "06095",  # Solano
    "06097",  # Sonoma
]

### Step 1 - import functions for data cleaning

these functions can bee found in `../analysis/functions.py`

In [3]:
import sys
sys.path.append("..")

from analysis import functions as fn

### Step 2 - produce cleaned LODES dataframes at census tract and county levels (using defined functions)


In [4]:
# NOTE this code cell took me 90 sec to run

# for year 2017
LODES_2017 = fn.load_and_clean_lodes_od(2017,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2017)
# for year 2018
LODES_2018 = fn.load_and_clean_lodes_od(2018,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2018)
# for year 2019
LODES_2019 = fn.load_and_clean_lodes_od(2019,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2019)
# for year 2020
LODES_2020 = fn.load_and_clean_lodes_od(2020,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2020)
# for year 2021
LODES_2021 = fn.load_and_clean_lodes_od(2021,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2021)
# for year 2022
LODES_2022 = fn.load_and_clean_lodes_od(2022,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2022)
# for year 2023
LODES_2023 = fn.load_and_clean_lodes_od(2023,lodes_path, bay_area_county_FIPS)
fn.lodes_county_tracts_adder(LODES_2023)

In [5]:
# convert dataframes from block-level to tract-level (func deletes block-level)
LODES_2017_tract = fn.lodes_tract_grouper(LODES_2017)
LODES_2018_tract = fn.lodes_tract_grouper(LODES_2018)
LODES_2019_tract = fn.lodes_tract_grouper(LODES_2019)
LODES_2020_tract = fn.lodes_tract_grouper(LODES_2020)
LODES_2021_tract = fn.lodes_tract_grouper(LODES_2021)
LODES_2022_tract = fn.lodes_tract_grouper(LODES_2022)
LODES_2023_tract = fn.lodes_tract_grouper(LODES_2023)

In [6]:
# make new county-level dfs
LODES_2017_county = fn.lodes_county_grouper(LODES_2017_tract)
LODES_2018_county = fn.lodes_county_grouper(LODES_2018_tract)
LODES_2019_county = fn.lodes_county_grouper(LODES_2019_tract)
LODES_2020_county = fn.lodes_county_grouper(LODES_2020_tract)
LODES_2021_county = fn.lodes_county_grouper(LODES_2021_tract)
LODES_2022_county = fn.lodes_county_grouper(LODES_2022_tract)
LODES_2023_county = fn.lodes_county_grouper(LODES_2023_tract) 

### Step 3 - mapping tracts to cities using shapefiles

Step 3 part A - matching tracts to cities

In [7]:
# import tract geometry
# idk why but this always gives an error message the first time but not if you run it again
tracts_path = boundaries_path+'/'+"tl_2023_06_tract.zip"

CA_tracts = gpd.read_file(tracts_path)
CA_tracts["county_fips"] = "06" + CA_tracts["COUNTYFP"] # done to match format of bay_area_county_FIPS entries
# filter to only bay area tracts
bay_tracts = CA_tracts[CA_tracts["county_fips"].isin(bay_area_county_FIPS)].copy()
bay_tracts = bay_tracts[["GEOID", "county_fips", "NAMELSAD", "geometry"]].rename(columns={'GEOID':'tract_geoid',"NAMELSAD":'tract_name'}) # NAMELSAD column is the tract name in the format "Census Tract ####"

del CA_tracts
gc.collect()

0

In [8]:
# import city administartive geometry
places_path = boundaries_path+'/'+"tl_2023_06_place.zip"

CA_places = gpd.read_file(places_path)[["GEOID", "NAME", "NAMELSAD", "geometry"]].rename(columns={'GEOID':'place_geoid','NAME':'city_name'})

In [9]:
# reproject into common crs
# Reproject to EPSG:2227 (this epsg was used in lab 9)
bay_tracts = bay_tracts.to_crs(epsg=2227)
CA_places = CA_places.to_crs(epsg=2227)

In [10]:
tract_to_city = fn.place_mapper_dict_maker(bay_tracts,CA_places)

c:\Users\gpapa\projects\Group_18_TrackB_midterm\.venv\Lib\site-packages\geopandas\tools\overlay.py:386: UserWarning: `keep_geom_type=True` in overlay resulted in 1801 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


Step 3 part B - applying the mapping to the LODES dataset

In [11]:
LODES_2017_city = fn.lodes_city_adder_grouper(LODES_2017_tract, tract_to_city)
LODES_2018_city = fn.lodes_city_adder_grouper(LODES_2018_tract, tract_to_city)
LODES_2019_city = fn.lodes_city_adder_grouper(LODES_2019_tract, tract_to_city)
LODES_2020_city = fn.lodes_city_adder_grouper(LODES_2020_tract, tract_to_city)
LODES_2021_city = fn.lodes_city_adder_grouper(LODES_2021_tract, tract_to_city)
LODES_2022_city = fn.lodes_city_adder_grouper(LODES_2022_tract, tract_to_city)
LODES_2023_city = fn.lodes_city_adder_grouper(LODES_2023_tract, tract_to_city)

### Step 4 - Exporting cleaned dataframes

In [12]:
# change FIPS codes to county names
county_fips_to_name = {"06001":"Alameda", 
                       "06013":'Contra Costa', 
                       "06041":'Marin', 
                       "06055":'Napa', 
                       "06075": 'San Francisco', 
                       "06081": 'San Mateo', 
                       "06085":'Santa Clara', 
                       "06095": 'Solano', 
                       "06097":'Sonoma'}



In [13]:
# change the county columns from fips codes to county names
LODES_2017_county = fn.county_fips_to_name_mapper(LODES_2017_county, county_fips_to_name)
LODES_2018_county = fn.county_fips_to_name_mapper(LODES_2018_county, county_fips_to_name)
LODES_2019_county = fn.county_fips_to_name_mapper(LODES_2019_county, county_fips_to_name)
LODES_2020_county = fn.county_fips_to_name_mapper(LODES_2020_county, county_fips_to_name)
LODES_2021_county = fn.county_fips_to_name_mapper(LODES_2021_county, county_fips_to_name)
LODES_2022_county = fn.county_fips_to_name_mapper(LODES_2022_county, county_fips_to_name)
LODES_2023_county = fn.county_fips_to_name_mapper(LODES_2023_county, county_fips_to_name)

In [14]:
# now export everything into csvs

LODES_2017_tract.to_csv((processed_path+'/LODES_2017_tract.csv'), index=False)
LODES_2017_county.to_csv((processed_path+'/LODES_2017_county.csv'), index=False)
LODES_2017_city.to_csv((processed_path+'/LODES_2017_city.csv'), index=False)

LODES_2018_tract.to_csv((processed_path+'/LODES_2018_tract.csv'), index=False)
LODES_2018_county.to_csv((processed_path+'/LODES_2018_county.csv'), index=False)
LODES_2018_city.to_csv((processed_path+'/LODES_2018_city.csv'), index=False)

LODES_2019_tract.to_csv((processed_path+'/LODES_2019_tract.csv'), index=False)
LODES_2019_county.to_csv((processed_path+'/LODES_2019_county.csv'), index=False)
LODES_2019_city.to_csv((processed_path+'/LODES_2019_city.csv'), index=False)

LODES_2020_tract.to_csv((processed_path+'/LODES_2020_tract.csv'), index=False)
LODES_2020_county.to_csv((processed_path+'/LODES_2020_county.csv'), index=False)
LODES_2020_city.to_csv((processed_path+'/LODES_2020_city.csv'), index=False)

LODES_2021_tract.to_csv((processed_path+'/LODES_2021_tract.csv'), index=False)
LODES_2021_county.to_csv((processed_path+'/LODES_2021_county.csv'), index=False)
LODES_2021_city.to_csv((processed_path+'/LODES_2021_city.csv'), index=False)

LODES_2022_tract.to_csv((processed_path+'/LODES_2022_tract.csv'), index=False)
LODES_2022_county.to_csv((processed_path+'/LODES_2022_county.csv'), index=False)
LODES_2022_city.to_csv((processed_path+'/LODES_2022_city.csv'), index=False)

LODES_2023_tract.to_csv((processed_path+'/LODES_2023_tract.csv'), index=False)
LODES_2023_county.to_csv((processed_path+'/LODES_2023_county.csv'), index=False)
LODES_2023_city.to_csv((processed_path+'/LODES_2023_city.csv'), index=False)

### Step 5 - Logging for report

In [15]:
# summary of what has been done to the data, for just one year
# manual change: if you want summary for a different year swap all instances of '2023' with that year 

raw_lodes_file = lodes_path+f"/ca_od_main_JT00_2023.csv.gz"
raw_s000_series = pd.read_csv(raw_lodes_file,
    compression="gzip",
    usecols=["S000"],
    dtype={"S000": int})["S000"]

raw_lodes_rows = len(raw_s000_series)
raw_lodes_jobs = int(raw_s000_series.sum())

del raw_s000_series
gc.collect()

bay_block_rows = len(LODES_2023)

tract_od_rows = len(LODES_2023_tract)
tract_commuters = int(LODES_2023_tract["commuters"].sum())

city_od_rows = len(LODES_2023_city)
city_commuters = int(LODES_2023_city["commuters"].sum())

county_od_rows = len(LODES_2023_county)
county_commuters = int(LODES_2023_county["commuters"].sum())

dropped_lodes_rows = raw_lodes_rows - bay_block_rows

ca_tracts_total = 9128  # Total 2023 CA tracts from Census TIGER/Line, hardcoded for now, will fix later
bay_tracts_total = len(bay_tracts)
dropped_ca_tracts = ca_tracts_total - bay_tracts_total

unincorp_tracts = int((CA_places["city_name"] == "Unincorporated").sum())
incorp_tracts = bay_tracts_total - unincorp_tracts

unique_cities = int(CA_places[CA_places["city_name"] != "Unincorporated"]["city_name"].nunique())

print(f'DATA CLEANING LOG - LODES 2023')
print('')
print('LODES commute data')
print(f"Raw California OD pairs (LODES8 2023 JT00):  {raw_lodes_rows} rows; {raw_lodes_jobs} jobs")
print(f"Dropped non-bay area records:                {dropped_lodes_rows} rows")
print(f"Retained bay area block-level OD pairs:      {bay_block_rows} rows; {tract_commuters} jobs")
print('')
print('Spatial granularity grouping')
print(f"Block-to-tract grouping:   {bay_block_rows} block pairs -> {tract_od_rows} tract pairs")
print(f"Tract-to-city grouping:    {tract_od_rows} tract pairs -> {city_od_rows} city pairs")
print(f"Tract-to-county grouping:  {tract_od_rows} tract pairs -> {county_od_rows} county pairs")
print('')
print('Spatial overlay for city grouping')
print(f"Total California census tracts:             {ca_tracts_total} tracts")
print(f"9-County bay area tracts only:              {bay_tracts_total} tracts")
print(f"Dropped non-bay area tracts:                {dropped_ca_tracts} tracts")
print(f"Projected coordinate reference system:      EPSG:2227 (NAD83 / CA State Plane Zone 3, US Feet)")
print(f"Tract-to-city assignment method:            Largest-Share rule from Lab 9 using gpd.overlay(how='intersection')")
print(f" -> Tracts assigned to official city/CDPs:     {incorp_tracts} tracts across {unique_cities} unique municipalities")
print(f" -> Tracts assigned to unincorporated county:  {unincorp_tracts} tracts")
print('')
print('Check to make sure no commuters were accidentally dropped')
print(f"Total commuters at tract level:  {tract_commuters}")
print(f"Total commuters at city level:   {city_commuters}")
print(f"Total commuters at county level: {county_commuters}")
print(f"No commuters lost? {tract_commuters == city_commuters == county_commuters}")

DATA CLEANING LOG - LODES 2023

LODES commute data
Raw California OD pairs (LODES8 2023 JT00):  16241068 rows; 17634412 jobs
Dropped non-bay area records:                13269133 rows
Retained bay area block-level OD pairs:      2971935 rows; 3288681 jobs

Spatial granularity grouping
Block-to-tract grouping:   2971935 block pairs -> 767054 tract pairs
Tract-to-city grouping:    767054 tract pairs -> 22336 city pairs
Tract-to-county grouping:  767054 tract pairs -> 81 county pairs

Spatial overlay for city grouping
Total California census tracts:             9128 tracts
9-County bay area tracts only:              1772 tracts
Dropped non-bay area tracts:                7356 tracts
Projected coordinate reference system:      EPSG:2227 (NAD83 / CA State Plane Zone 3, US Feet)
Tract-to-city assignment method:            Largest-Share rule from Lab 9 using gpd.overlay(how='intersection')
 -> Tracts assigned to official city/CDPs:     1772 tracts across 1594 unique municipalities
 -> Tracts 

## SECTION 2 - ACS data cleaning

Work-from-home rates from ACS 1-year samples 
Spatial granularity: the county level 
Time series: for all years excpet 2020 (2017-2019 & 2021-2023)

### Step 0 - set up directory paths

In [16]:
# folder paths
acs_path = "../data/raw/acs" # followed by the year folder, then the file "ACSDT1Y[year].B08301-Data.csv"

bay_area_county_FIPS = ["06001",  # Alameda
                        "06013",  # Contra Costa
                        "06041",  # Marin
                        "06055",  # Napa
                        "06075",  # San Francisco
                        "06081",  # San Mateo
                        "06085",  # Santa Clara
                        "06095",  # Solano
                        "06097",  # Sonoma
                       ]

county_fips_to_name = {"06001":"Alameda", 
                       "06013":'Contra Costa', 
                       "06041":'Marin', 
                       "06055":'Napa', 
                       "06075": 'San Francisco', 
                       "06081": 'San Mateo', 
                       "06085":'Santa Clara', 
                       "06095": 'Solano', 
                       "06097":'Sonoma'}

### Step 1 - Clean, raw data, Calculate WFH, produce dataframe to export

A requirement of the midterm submission for this project is the turn repeated code into functions. However, we are unfamiliar with how to vectorize code snippets that involve changing variable names, as seen below. This will be discussed in the challenges section of our report.

In [17]:
list_of_years = ['2017','2018','2019','2021','2022','2023']
acs_filepaths = np.array([])
for yr in list_of_years:
    acs_filepaths = np.append(acs_filepaths, acs_path+"/"+yr+"/ACSDT1Y"+yr+".B08301-Data.csv")

In [18]:
acs_2017_raw = pd.read_csv(acs_filepaths[0])
acs_2018_raw = pd.read_csv(acs_filepaths[1])
acs_2019_raw = pd.read_csv(acs_filepaths[2])
# acs_2020 DNE
acs_2021_raw = pd.read_csv(acs_filepaths[3])
acs_2022_raw = pd.read_csv(acs_filepaths[4])
acs_2023_raw = pd.read_csv(acs_filepaths[5])

In [19]:
# remove the 'margin of error' columns and the 
# remove first row (column descriptors, we can get that from the metadata csv)

acs_2017 = fn.acs_col_desc_cleaner(acs_2017_raw)
acs_2018 = fn.acs_col_desc_cleaner(acs_2018_raw)
acs_2019 = fn.acs_col_desc_cleaner(acs_2019_raw)
acs_2021 = fn.acs_col_desc_cleaner(acs_2021_raw)
acs_2022 = fn.acs_col_desc_cleaner(acs_2022_raw)
acs_2023 = fn.acs_col_desc_cleaner(acs_2023_raw)

There are some missing values in the dataset for Napa county, which was was discovered by manually checking. for now we deal with it by hardcoding a solution; later we will write functions to fix this missing value problem.

In [20]:
# Napa county is missing in the 2021 and 2022 datasets, so we insert a row of 1.0 as a placeholder.
nan_list = ['0500000US06055','Napa County, California'] + np.ones(len(acs_2017.columns)-2).tolist()
nan_row = pd.DataFrame([nan_list], columns=acs_2017.columns)
acs_2021 = pd.concat([acs_2021.iloc[:3], nan_row, acs_2021.iloc[3:]]).reset_index(drop=True)
acs_2022 = pd.concat([acs_2022.iloc[:3], nan_row, acs_2022.iloc[3:]]).reset_index(drop=True)
# ONLY RUN ONCE!

In [21]:
# add a column to the acs datasets that includes the county names
acs_2017 = fn.acs_county_extractor(acs_2017, county_fips_to_name)
acs_2018 = fn.acs_county_extractor(acs_2018, county_fips_to_name)
acs_2019 = fn.acs_county_extractor(acs_2019, county_fips_to_name)
# no 2020
acs_2021 = fn.acs_county_extractor(acs_2021, county_fips_to_name)
acs_2022 = fn.acs_county_extractor(acs_2022, county_fips_to_name)
acs_2023 = fn.acs_county_extractor(acs_2023, county_fips_to_name)

In [22]:
# make the dataframes that have work from home proportions for each county
acs_2017_wfh, prop_wfh_2017 = fn.acs_wfh_adder(acs_2017)
acs_2018_wfh, prop_wfh_2018 = fn.acs_wfh_adder(acs_2018)
acs_2019_wfh, prop_wfh_2019 = fn.acs_wfh_adder(acs_2019)
# no 2020
acs_2021_wfh, prop_wfh_2021 = fn.acs_wfh_adder(acs_2021)
acs_2022_wfh, prop_wfh_2022 = fn.acs_wfh_adder(acs_2022)
acs_2023_wfh, prop_wfh_2023 = fn.acs_wfh_adder(acs_2023)

In [23]:
# then make it into one df

acs_wfh = pd.DataFrame()
acs_wfh['FIPS'] = acs_2017_wfh['FIPS'].to_numpy()
acs_wfh['county'] = acs_2017_wfh['county'].to_numpy()

acs_wfh['WFH prop 2017'] = prop_wfh_2017.to_numpy()
acs_wfh['WFH prop 2018'] = prop_wfh_2018.to_numpy()
acs_wfh['WFH prop 2019'] = prop_wfh_2019.to_numpy()

acs_wfh['WFH prop 2021'] = prop_wfh_2021.to_numpy()
acs_wfh['WFH prop 2022'] = prop_wfh_2022.to_numpy()
acs_wfh['WFH prop 2023'] = prop_wfh_2023.to_numpy()

In [24]:
# replace Napa county missing entries with nan instead of 1
acs_wfh = acs_wfh.replace(1.0, np.nan)

### Step 2 - Export

In [26]:
# export csv
acs_wfh.to_csv((processed_path+'/ACS_WFH.csv'), index=False)

### Step 3 - Logging

In [27]:
# data cleaning log code
num_M_cols = acs_2017_raw.loc[:, acs_2017_raw.columns.str.contains('M$')].shape[1]

In [28]:
print('DATA CLEANING LOG - ACS WORK-FROM-HOME (2017--2023 1-YR SAMPLES, EXCLUDING 2020)')
print('')
print("Raw files from table B08301")
print("Raw census file dimensions per year:")
print(f" - 2017: {acs_2017_raw.shape[0]} rows, {acs_2017_raw.shape[1]} columns")
print(f" - 2018: {acs_2018_raw.shape[0]} rows, {acs_2018_raw.shape[1]} columns")
print(f" - 2019: {acs_2019_raw.shape[0]} rows, {acs_2019_raw.shape[1]} columns")
print(f" - 2021: {acs_2021_raw.shape[0]} rows, {acs_2021_raw.shape[1]} columns")
print(f" - 2022: {acs_2022_raw.shape[0]} rows, {acs_2022_raw.shape[1]} columns")
print(f" - 2023: {acs_2023_raw.shape[0]} rows, {acs_2023_raw.shape[1]} columns")
print('')
print("Cleaning & standardization (also per year)")
print("Metadata Stripping:           Removed Row 1 ('Geography' text header)")
print(f"Column Pruning:               Dropped all 'M$' margin of error columns (number of cols = {num_M_cols})")
print("Geographic Key Alignment:     Sliced GEO_ID to 5-digit county FIPS (06001 - 06097)")
print("County Name Mapping:          Mapped 9 Bay Area FIPS codes to county string names")
print("Indicator Formulation:        WFH Proportion = B08301_021E / B08301_001E")
print('')
print("Handling missing values for Napa county")
print("Napa County (FIPS = 06055) NaNs:    2 years: 2021 and 2022 (determined by inspection)") 
print("Solution:    Preserved as NaN (omitted by Census due to small sample sizes for those years)")
print('')
print("Dimensions of cleaned dataframe")
print(f"Number of rows (counties):     {acs_wfh.shape[0]}")
print(f"Number of columns:             {acs_wfh.shape[1]}")
print(f"Column labels:                 {acs_wfh.columns}")
print(f"Destination Path:              {processed_path}/ACS_WFH.csv")


DATA CLEANING LOG - ACS WORK-FROM-HOME (2017--2023 1-YR SAMPLES, EXCLUDING 2020)

Raw files from table B08301
Raw census file dimensions per year:
 - 2017: 10 rows, 45 columns
 - 2018: 10 rows, 45 columns
 - 2019: 10 rows, 45 columns
 - 2021: 9 rows, 45 columns
 - 2022: 9 rows, 45 columns
 - 2023: 10 rows, 45 columns

Cleaning & standardization (also per year)
Metadata Stripping:           Removed Row 1 ('Geography' text header)
Column Pruning:               Dropped all 'M$' margin of error columns (number of cols = 21)
Geographic Key Alignment:     Sliced GEO_ID to 5-digit county FIPS (06001 - 06097)
County Name Mapping:          Mapped 9 Bay Area FIPS codes to county string names
Indicator Formulation:        WFH Proportion = B08301_021E / B08301_001E

Handling missing values for Napa county
Napa County (FIPS = 06055) NaNs:    2 years: 2021 and 2022 (determined by inspection)
Solution:    Preserved as NaN (omitted by Census due to small sample sizes for those years)

Dimensions of cl